# Event sparsity of the validation recordings

How little of each recording is actually the target species. Per file, then per subset.

**Input:** `Development_Set/Validation_Set` — the annotation csvs, plus each wav's header
for its duration. **No audio samples are decoded**, so the whole notebook runs in seconds.

**Outputs:** two csv files, two LaTeX tables in `paper/tables/`, one figure in
`paper/figures/`.

---

## The three numbers

Everything comes from the **POS** intervals in each csv.

| quantity | definition |
|---|---|
| **Coverage** | share of the recording's duration that lies inside a positive event, as a percentage |
| **Sparsity** | $100 - \text{coverage}$ — the share of the recording that is *not* the target |
| **Event rate** | positive events per minute of recording |
| **Median gap** | median silent stretch between one event ending and the next beginning |

Coverage and rate say different things, and a recording can be sparse in one sense and not
the other: many short chirps give a high rate with low coverage, while a single long bout
gives a low rate with high coverage. Quote both.

Overlapping annotations are **merged** before coverage is summed, so coverage can never
exceed 100%; the event *count* and median event duration use the annotations as written,
since those are the biological units. UNK and NEG intervals are excluded from coverage, and
UNK coverage is reported separately so it is visible rather than silently dropped.

## Why it matters for this task

Coverage is the prior probability that a randomly chosen frame is a positive. At 1%
coverage a detector sees 99 negative frames for every positive one, so even a small
per-frame false-positive rate produces far more false detections than there are true events
— which is exactly the failure mode the per-file precision numbers show.


## 0 — Configuration

In [ ]:
import os
from pathlib import Path

# configs/train.yaml -> path.eval_dir. Override without editing:  export DCASE_DATA_ROOT=...
DATA_ROOT = Path(os.environ.get("DCASE_DATA_ROOT", "/media/helios/Data/Uni/Masters/Dataset"))
VAL_DIR = DATA_ROOT / "Development_Set" / "Validation_Set"

# Coverage runs below 1% on some files, so two decimals are needed.
N_DECIMALS = 2

# Your per-subset result tables show five subsets, because PB and PB24 merge when
# notebooks/file_to_subset.json is absent -- both hold BUK* files. Set
# MERGE_SUBSETS = {"PB24": "PB"} to make this table line up with them row for row.
MERGE_SUBSETS = {}

OUT_DIR = Path("sparsity")
PAPER_TABLES_DIR = Path("../../../paper/tables").resolve()
PAPER_FIGURES_DIR = Path("../../../paper/figures").resolve()
for d in (OUT_DIR, PAPER_TABLES_DIR, PAPER_FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"validation set : {VAL_DIR}\n                 exists: {VAL_DIR.is_dir()}")
print(f"tables         : {PAPER_TABLES_DIR}")
print(f"figures        : {PAPER_FIGURES_DIR}")

## 1 — Helpers

In [ ]:
import traceback

import numpy as np
import pandas as pd
import soundfile as sf

META_COLS = ["Audiofilename", "Starttime", "Endtime"]


def read_annotations(csv_path):
    """Return (pos, other): [start, end] arrays in seconds.

    Label columns are everything that is not a META_COL, so this reads both csv layouts in
    the dataset -- the validation set's single 'Q' column and the training set's
    one-column-per-class form (the pattern used in single_file_analysis.ipynb). A row is POS
    if any label column says POS; every other annotated row (UNK, NEG) goes to `other`.
    """
    df = pd.read_csv(csv_path, header=0, index_col=False)
    df.columns = [str(c).strip() for c in df.columns]
    class_cols = [c for c in df.columns if c not in META_COLS]
    labels = df[class_cols].astype(str).apply(lambda s: s.str.strip().str.upper())
    is_pos = (labels == "POS").any(axis=1).to_numpy()
    iv = df[["Starttime", "Endtime"]].astype(float).to_numpy()
    return iv[is_pos], iv[~is_pos]


def merge_intervals(iv):
    """Merge overlapping or touching [start, end] intervals into a disjoint, sorted set.

    Without this, overlapping annotations would be double-counted and coverage could
    exceed 100%.
    """
    if len(iv) == 0:
        return np.empty((0, 2))
    iv = iv[np.argsort(iv[:, 0])]
    out = [iv[0].astype(float).copy()]
    for s, e in iv[1:]:
        if s <= out[-1][1]:                     # overlaps or touches the open interval
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append(np.array([s, e], float))
    return np.array(out)


def gaps_between(merged):
    """Silent stretches between consecutive merged events (not the lead-in or the tail)."""
    return merged[1:, 0] - merged[:-1, 1] if len(merged) > 1 else np.empty(0)


def measure(wav_path, csv_path):
    """Sparsity of one file. Reads the wav header only -- no samples are decoded."""
    info = sf.info(str(wav_path))               # header read: duration, sr, no audio
    duration = info.duration
    pos, other = read_annotations(csv_path)

    merged = merge_intervals(pos)
    event_time = float((merged[:, 1] - merged[:, 0]).sum()) if len(merged) else 0.0
    unk_time = float(np.diff(merge_intervals(other), axis=1).sum()) if len(other) else 0.0
    gaps = gaps_between(merged)
    durations = pos[:, 1] - pos[:, 0] if len(pos) else np.empty(0)

    flags = []
    if len(pos) == 0:
        flags.append("no POS events")
    if len(merged) < len(pos):
        flags.append(f"{len(pos) - len(merged)} overlapping annotation(s) merged")
    if len(pos) and pos[:, 1].max() > duration + 0.05:
        flags.append(f"annotation ends {pos[:, 1].max() - duration:.1f} s past the audio")

    return {
        "filename": wav_path.name,
        "duration_s": round(duration, 1),
        "n_events": len(pos),
        "event_time_s": round(event_time, 2),
        "coverage_pct": round(100 * event_time / duration, 3) if duration else np.nan,
        "sparsity_pct": round(100 - 100 * event_time / duration, 3) if duration else np.nan,
        "events_per_min": round(len(pos) / (duration / 60), 2) if duration else np.nan,
        "event_dur_median_s": round(float(np.median(durations)), 3) if len(durations) else np.nan,
        "event_dur_min_s": round(float(durations.min()), 3) if len(durations) else np.nan,
        "event_dur_max_s": round(float(durations.max()), 3) if len(durations) else np.nan,
        "gap_median_s": round(float(np.median(gaps)), 2) if len(gaps) else np.nan,
        "gap_max_s": round(float(gaps.max()), 2) if len(gaps) else np.nan,
        "unk_coverage_pct": round(100 * unk_time / duration, 3) if duration else np.nan,
        "flag": "; ".join(flags),
    }

## 2 — Measure every validation file

The **subset** is the folder holding the wav (`.../Validation_Set/ME/ME1.wav` → `ME`),
which is how the DCASE evaluator derives subsets, so these labels match your result
tables.

In [ ]:
tasks, orphans = [], []
for wav in sorted(VAL_DIR.rglob("*.wav")):
    csv = wav.with_suffix(".csv")
    if csv.exists():
        tasks.append((MERGE_SUBSETS.get(wav.parent.name, wav.parent.name), wav, csv))
    else:
        orphans.append(wav)

if not tasks:
    raise RuntimeError(f"No wav/csv pairs under {VAL_DIR} -- check DATA_ROOT.")
if orphans:
    print(f"{len(orphans)} wav(s) have no csv and were skipped:")
    for wav in orphans:
        print(f"   {wav}")

rows, failed = [], []
for subset, wav, csv in tasks:
    try:
        rows.append({"subset": subset, **measure(wav, csv), "path": str(wav)})
    except Exception as exc:
        failed.append((wav.name, f"{type(exc).__name__}: {exc}", traceback.format_exc()))

# Reported before the table is built: if everything failed, `rows` is empty and any column
# access below would raise something unrelated to the real cause.
if failed:
    print(f"{len(failed)} of {len(tasks)} file(s) failed:")
    for name, err, _ in failed[:10]:
        print(f"   {name}: {err}")
    print(f"\nFull traceback for the first failure ({failed[0][0]}):\n")
    print(failed[0][2])
if not rows:
    raise RuntimeError(f"All {len(tasks)} file(s) failed -- see the traceback above.")

per_file = (pd.DataFrame(rows)
            .sort_values(["subset", "coverage_pct"], ignore_index=True))
per_file.to_csv(OUT_DIR / "validation_sparsity_per_file.csv", index=False)

print(f"{len(per_file)} files in {per_file['subset'].nunique()} subsets "
      f"-> {OUT_DIR / 'validation_sparsity_per_file.csv'}")
print(f"\ncoverage      median {per_file['coverage_pct'].median():.2f}%   "
      f"range {per_file['coverage_pct'].min():.2f}% to {per_file['coverage_pct'].max():.2f}%")
print(f"events/min    median {per_file['events_per_min'].median():.2f}   "
      f"range {per_file['events_per_min'].min():.2f} to {per_file['events_per_min'].max():.2f}")
print(f"event length  median {per_file['event_dur_median_s'].median():.2f} s   "
      f"shortest {per_file['event_dur_min_s'].min():.3f} s")
print(f"total         {int(per_file['n_events'].sum())} events over "
      f"{per_file['duration_s'].sum() / 3600:.1f} h of audio")

flagged = per_file[per_file["flag"] != ""]
if len(flagged):
    print(f"\n{len(flagged)} file(s) flagged:")
    for r in flagged.itertuples():
        print(f"   {r.filename:<40} {r.flag}")

## 3 — Table 1: per file

Sparsest first within each subset. The csv written above keeps the columns left out here
(min/max event duration, max gap, UNK coverage).

In [ ]:
def save_tex(df, filename, caption, label, **to_latex_kwargs):
    """Write `df` as a standalone LaTeX table -- same helper as the other paper notebooks."""
    kwargs = dict(float_format=f"%.{N_DECIMALS}f", na_rep="—", caption=caption, label=label)
    kwargs.update(to_latex_kwargs)
    out = PAPER_TABLES_DIR / filename
    out.write_text(df.to_latex(**kwargs))
    print(f"  saved {out}")


table_file = (per_file[["subset", "filename", "duration_s", "n_events", "events_per_min",
                        "event_dur_median_s", "coverage_pct", "gap_median_s"]]
              .rename(columns={"subset": "Subset", "filename": "File",
                               "duration_s": "Duration (s)", "n_events": "Events",
                               "events_per_min": "Events/min",
                               "event_dur_median_s": "Median event (s)",
                               "coverage_pct": "Coverage (\\%)",
                               "gap_median_s": "Median gap (s)"}))

save_tex(table_file, "sparsity_per_file.tex",
         caption=("Event sparsity of each recording in the DCASE 2024 Task~5 validation "
                  "set. Coverage is the share of the recording's duration lying inside an "
                  "annotated positive event, with overlapping annotations merged; the "
                  "median gap is the median silent interval between consecutive events. "
                  "Recordings are listed sparsest first within each subset."),
         label="tab:sparsity_per_file", index=False)

table_file.round(N_DECIMALS)

## 4 — Table 2: per subset

The one for the paper body. Medians across the recordings of each subset — with a handful
of files per subset, one unusual recording would drag a mean a long way. `Coverage (%)` is
also given pooled (total event time over total audio), which is the number to quote when
talking about the class balance a detector faces.

In [ ]:
agg = (per_file.groupby("subset")
       .agg(Files=("filename", "size"),
            Events=("n_events", "sum"),
            Hours=("duration_s", lambda s: s.sum() / 3600),
            EventHours=("event_time_s", lambda s: s.sum() / 3600),
            RateMed=("events_per_min", "median"),
            EventMed=("event_dur_median_s", "median"),
            CovMed=("coverage_pct", "median"),
            GapMed=("gap_median_s", "median")))
agg["CovPooled"] = 100 * agg["EventHours"] / agg["Hours"]
by_subset = (agg[["Files", "Events", "Hours", "RateMed", "EventMed", "CovMed",
                  "CovPooled", "GapMed"]]
             .sort_values("CovPooled"))
by_subset.index.name = "Subset"
by_subset.columns = ["Files", "Events", "Hours", "Events/min (med.)", "Median event (s)",
                     "Coverage \\% (med.)", "Coverage \\% (pooled)", "Median gap (s)"]

overall = pd.DataFrame([{
    "Files": len(per_file),
    "Events": int(per_file["n_events"].sum()),
    "Hours": per_file["duration_s"].sum() / 3600,
    "Events/min (med.)": per_file["events_per_min"].median(),
    "Median event (s)": per_file["event_dur_median_s"].median(),
    "Coverage \\% (med.)": per_file["coverage_pct"].median(),
    "Coverage \\% (pooled)": 100 * per_file["event_time_s"].sum() / per_file["duration_s"].sum(),
    "Median gap (s)": per_file["gap_median_s"].median(),
}], index=["All"])
overall.index.name = "Subset"

by_subset_out = pd.concat([by_subset, overall])
by_subset_out.to_csv(OUT_DIR / "validation_sparsity_by_subset.csv")

save_tex(by_subset_out, "sparsity_by_subset.tex",
         caption=("Event sparsity per subset of the DCASE 2024 Task~5 validation set. "
                  "Median columns summarise the per-recording values of each subset; "
                  "pooled coverage is total annotated event time over total audio "
                  "duration, i.e. the share of frames a detector sees as positive. "
                  "Subsets are ordered sparsest first. See "
                  "Table~\\ref{tab:sparsity_per_file} for individual recordings."),
         label="tab:sparsity_by_subset")

by_subset_out.round(N_DECIMALS)

## 5 — Figure

Left: how much of each recording is the target species (log scale — the values span more
than two orders of magnitude). Right: how often events occur. A file low on the left and
high on the right is many brief calls; low on both is a genuinely empty recording.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.alpha": 0.4, "font.size": 10,
})
BLUE, INK, GREY = "#2a78d6", "#3a3a3a", "#9a9a9a"


def save_fig(fig, stem, paper=True):
    fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", dpi=150)
    print(f"  saved {OUT_DIR / f'{stem}.png'}")
    if paper:
        fig.savefig(PAPER_FIGURES_DIR / f"{stem}.pdf", bbox_inches="tight")
        print(f"  saved {PAPER_FIGURES_DIR / f'{stem}.pdf'}")


order = list(by_subset.index)
rng = np.random.default_rng(1234)

fig, axes = plt.subplots(1, 2, figsize=(1.1 * len(order) + 7.0, 4.0))
panels = [("coverage_pct", "Share of recording that is the target (%)", True),
          ("events_per_min", "Events per minute", False)]

for ax, (col, ylabel, log) in zip(axes, panels):
    for i, subset in enumerate(order):
        vals = per_file.loc[per_file["subset"] == subset, col].dropna()
        if not len(vals):
            continue
        ax.scatter(i + rng.uniform(-0.14, 0.14, len(vals)), vals, s=34, color=BLUE,
                   alpha=0.85, edgecolor="white", linewidth=0.6, zorder=3)
        med = vals.median()
        ax.plot([i - 0.3, i + 0.3], [med, med], color=INK, lw=2, zorder=4)
        ax.annotate(f"{med:.2f}", (i + 0.33, med), fontsize=9, color=INK, va="center")
    if log:
        ax.set_yscale("log")
        ax.grid(True, which="minor", axis="y", alpha=0.15)
    ax.set_xticks(range(len(order)),
                  [f"{s}\n(n={int(by_subset.loc[s, 'Files'])})" for s in order])
    ax.set_xlim(-0.6, len(order) - 0.15)
    ax.set_xlabel("Subset (sparsest first)")
    ax.set_ylabel(ylabel)

axes[0].set_title("Event coverage", loc="left", fontsize=11)
axes[1].set_title("Event rate", loc="left", fontsize=11)
fig.tight_layout()
save_fig(fig, "sparsity_by_subset")
plt.show()

## Writing it up

A paragraph to adapt:

> Positive events are sparse in these recordings. For each file we take the annotated POS
> intervals, merge any that overlap, and report the resulting event time as a percentage of
> the file's duration (coverage), the number of annotated events per minute, and the median
> silent interval between consecutive events. Pooled over the validation set, annotated
> events occupy only *X*\% of the audio, so a frame drawn at random is a negative with
> probability *100-X*\%; the median event lasts *Y*~s while the median gap between events
> is *Z*~s.

Then the consequence, which is the reason the table is in the paper: at that coverage the
frame-level class balance is roughly 1:*N*, so a per-frame false-positive rate of even a few
percent yields more false detections than there are true events. That is the mechanism
behind the low precision in the results tables, and it differs by subset — which the
per-subset rows make visible.

**Definitional points to state**, since a reader can reasonably assume otherwise:

1. **Coverage uses merged intervals**, so overlapping annotations are not double-counted;
   the event *count* and median event length use the annotations as written.
2. **UNK and NEG intervals are excluded** from coverage. `unk_coverage_pct` in the csv shows
   how much time that removes per file — worth checking before claiming a coverage figure,
   because a file with substantial UNK time is neither as sparse nor as clean as it looks.
3. **Gaps are between consecutive events only** — the stretch before the first event and
   after the last are not counted, as they are bounded by the recording, not by the species.
4. **Median, not mean, per subset**, with the file count shown; means over a handful of
   recordings of very different length are easily dominated by one file.

**The `flag` column** records the two things that would otherwise distort a number quietly:
overlapping annotations that had to be merged, and annotations extending past the end of the
audio (a dataset inconsistency — that file's coverage is understated relative to its
annotation, and it is worth reporting as such rather than silently clipping).
